# End-to-end walkthrough — Revenue Prediction Accelerator

This notebook walks the **full lifecycle** end to end and ties together the
[demo script](../../docs/demo/end-to-end-demo-script.md), the
[operational runbooks](../../docs/operations/runbooks/README.md), and the
[four authoring patterns](../../docs/patterns/end-to-end-patterns.md).

**Sequence:** frame -> data -> train & compare -> evaluate -> KPI scorecard ->
explain -> batch score -> (cloud) AutoML + code-first -> register -> gate ->
deploy -> monitor + continuous evaluation.

> All data is **synthetic**; identifiers are placeholders. The offline cells run
> with no cloud. Cloud cells are **opt-in** and reference the Azure ML Studio UI,
> AutoML, and the code-first SDK.

**Prerequisites (offline):** `uv sync --extra api` · **cloud (opt-in):** `uv sync --extra azure` + `az login`.

## 0. The Build & Learn app (optional, parallel to this notebook)

The React **Build & Learn** UI teaches the same lifecycle interactively.

```bash
scripts/start-ui.sh      # -> http://127.0.0.1:8000  (API docs at /docs)
scripts/stop-ui.sh       # stop it when done
```

Run those in a terminal; this notebook uses the same core package directly.

## 1. Frame the problem & load configuration

Grain: `facility_id x accounting_month x snapshot_date`. Target:
`actual_month_end_net_revenue` (known only after close). Primary metric: **WAPE**.

In [1]:
from revenue_prediction.config.loader import load_settings

settings = load_settings("dev")
print("environment      :", settings.environment)
print("facilities       :", settings.data.n_facilities)
print("snapshot days    :", settings.data.snapshot_days)
print("candidates       :", settings.model.candidates)
print("primary metric   :", settings.model.primary_metric)

environment      : dev
facilities       : 6
snapshot days    : [7, 10, 12, 15, 18, 21, 24, 27]
candidates       : ['naive_prior', 'seasonal_naive', 'elastic_net', 'gradient_boosting', 'hist_gradient_boosting', 'xgboost']
primary metric   : wape


## 2. Generate the synthetic dataset

Equivalent to `uv run revenue-prediction generate-data`.

In [ ]:
from revenue_prediction.core.data.synthetic import generate_synthetic_dataset

frame = generate_synthetic_dataset(settings.data)
print(frame.shape)
frame.head()

## 3. Train & compare models (code-first)

Trains all candidates with **time-aware** validation and selects the champion by
**WAPE** — the same code the Azure ML command job runs
(`core.training.azureml_entry`). See
[models & metrics](../../docs/modeling/models-and-metrics.md) for what each model is.

In [ ]:
from revenue_prediction.pipelines.local_pipeline import run_local_pipeline
from revenue_prediction.core.evaluation import comparison_table

result = run_local_pipeline(settings, frame=frame)
table = comparison_table(result.results).sort_values("wape").reset_index(drop=True)
print("Champion  :", result.selection.champion)
print("Challenger:", result.selection.challenger)
table.round(4)

## 4. Evaluate — overall and disaggregated

A good average can hide a bad facility or a weak early checkpoint, so we report
**by facility** and **by snapshot day** too.

In [ ]:
from revenue_prediction.core.data.schema import TARGET
from revenue_prediction.core.evaluation import (
    metrics_by_facility,
    metrics_by_snapshot_day,
)
from revenue_prediction.core.inference.predict import batch_predict

PRED = "predicted_month_end_net_revenue"
keys = ["facility_id", "accounting_month", "snapshot_date", "snapshot_day"]
scored = batch_predict(result.champion_bundle, frame.drop(columns=[TARGET]))
eval_frame = scored.merge(frame[[*keys, TARGET]], on=keys)
print("By snapshot day:")
display(metrics_by_snapshot_day(eval_frame, TARGET, PRED).round(4))
print("By facility (head):")
display(metrics_by_facility(eval_frame, TARGET, PRED).round(4).head())

## 5. KPI scorecard — metrics vs. targets and the manual baseline

Grades the model against per-checkpoint WAPE targets and the beat-the-analyst KPI.
See [success metrics & KPIs](../../docs/modeling/success-metrics-and-kpis.md).
CLI equivalent: `uv run revenue-prediction scorecard <predictions> <actuals> --baseline-wape 0.08`.

In [ ]:
from revenue_prediction.core.evaluation import compute_metrics, kpi_scorecard

overall = compute_metrics(eval_frame[TARGET], eval_frame[PRED])
by_day = metrics_by_snapshot_day(eval_frame, TARGET, PRED)
kpi_scorecard(overall, by_day, baseline_wape=0.08)

## 6. Explainability — which drivers matter

Permutation importance (model-agnostic). AutoML produces equivalent explanations
automatically in the Studio UI. See
[responsible AI](../../docs/governance/responsible-ai.md).

In [ ]:
from revenue_prediction.core.data.schema import FEATURE_COLUMNS
from revenue_prediction.core.evaluation.explainability import permutation_feature_importance

# Explain the champion; if it is a naive baseline, fall back to the top learned model.
champ = result.results[result.selection.champion]
if champ.feature_builder is None:
    for name in result.selection.ranking["model"].tolist():
        candidate = result.results[str(name)]
        if candidate.feature_builder is not None:
            champ = candidate
            break

if champ.feature_builder is not None:
    raw = [c for c in FEATURE_COLUMNS if c in frame.columns]
    x_matrix = champ.feature_builder.transform(frame[raw])
    importance = permutation_feature_importance(champ.estimator, x_matrix, frame[TARGET], n_repeats=5)
    display(importance.head(10))
else:
    print("No learned model with a feature builder to explain.")

## 7. Batch inference on a mid-month checkpoint

Inference never needs the target. `cutoff_day` restricts to an as-of checkpoint.
CLI: `uv run revenue-prediction predict <bundle> <snapshot> --cutoff-day 15`.

In [ ]:
checkpoint = frame[frame["snapshot_day"] == 15].drop(columns=[TARGET])
predictions = batch_predict(result.champion_bundle, checkpoint, cutoff_day=15)
predictions[["facility_id", "accounting_month", "snapshot_day", PRED, "model_name", "cutoff_day"]].head()

## 8. Cloud sequencing (opt-in) — AutoML, code-first, register, deploy

The cells below mirror the [deployment guide](../../docs/deployment/guide.md) and
the [runbooks](../../docs/operations/runbooks/README.md). They require
`uv sync --extra azure` and `az login`. Prefer the **Studio UI** for patterns 1 & 4
and the **SDK** for patterns 2 & 3.

### 8a. Connect + register data and environment

```python
from revenue_prediction.integrations.azureml.client import get_ml_client
client = get_ml_client(settings.azure_ml)
# Register the Parquet (code-first) and MLTable (AutoML) data assets, and the
# environment from mlops/environments/environment.yml. See notebooks/code_first/
# and notebooks/automl/ for ready snippets.
```

### 8b. Pattern 2 — AutoML via the SDK (benchmark)

```python
from revenue_prediction.integrations.automl import build_automl_job_spec, build_regression_job
spec = build_automl_job_spec(settings.automl, settings.azure_ml,
                             training_data_asset="azureml:revenue_snapshots_mltable@latest")
job = build_regression_job(spec)              # enable_model_explainability=True
automl_run = client.jobs.create_or_update(job)
# In Studio: open the job -> Models leaderboard -> best model -> Metrics + Explanations.
```

### 8c. Pattern 3 — code-first via the SDK (champion)

```python
from revenue_prediction.integrations.azureml.jobs import build_command_job, register_model_from_run
job = build_command_job(settings.azure_ml,
                        training_data_asset="azureml:revenue_snapshots@latest",
                        environment="azureml:revenue-prediction-env@latest")
run = client.jobs.create_or_update(job)
client.jobs.stream(run.name)
# Register the champion, tagged by authoring pattern so both paths show in Models:
register_model_from_run(client, run.name, settings.azure_ml, authoring_pattern="code_first")
```

> Patterns **1 (AutoML wizard)** and **4 (code-first in the UI/Designer)** do the
> same thing in the Studio UI — see [patterns](../../docs/patterns/end-to-end-patterns.md).

### 8d. Gate, deploy (batch), monitor + continuously evaluate

- **Gate:** champion/challenger + human approval — [model governance](../../docs/governance/model-governance.md).
- **Deploy:** pipeline-component batch deployment — [batch endpoint runbook](../../docs/operations/runbooks/batch-endpoint-deploy-and-update.md).
- **Monitor + continuous evaluation:** drift + predictions-vs-actuals WAPE/bias — [monitoring](../../docs/operations/monitoring.md).
- **Promote dev -> test -> prod:** [promotion runbook](../../docs/operations/runbooks/promote-across-environments.md).

## 9. Where to go next

- [End-to-end demo script](../../docs/demo/end-to-end-demo-script.md)
- [Model selection & evaluation across patterns](../../docs/patterns/model-selection-and-evaluation.md)
- [Models & metrics catalog](../../docs/modeling/models-and-metrics.md)
- [Operational runbooks](../../docs/operations/runbooks/README.md)
- Pattern notebooks: [`../automl/`](../automl/01_automl_regression.ipynb), [`../code_first/`](../code_first/01_code_first_training.ipynb), [`../responsible_ai/`](../responsible_ai/01_explainability_and_fairness.ipynb), [`../fabric/`](../fabric/01_onelake_predictions.ipynb)